[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P04/P04-1_Monte_Carlo.ipynb)

# Monte-Carlo avec NumPy

Chapitre 4 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre pièce découpée est percée, ajourée, chanfreinée, et vous voudriez connaître son aire ou son volume sans décomposer le dessin en figures élémentaires ? Quelle confiance accorder à ce chiffre ?

**Objectif terminal.** Ce chapitre montre comment tirer des points au hasard dans une boîte qui englobe la pièce, compter ceux qui tombent dans la matière, en déduire l'aire ou le volume, puis chiffrer l'incertitude. C'est la méthode de Monte-Carlo, que NumPy, la bibliothèque de calcul sur tableaux de Python, rend rapide.

**Objectif instrumental.**

- Créer un tableau NumPy et calculer sur tous ses éléments sans boucle.
- Tirer des nombres au hasard avec une graine fixée.
- Sélectionner par un masque de booléens, et éviter le piège de la vue.
- Estimer une aire ou un volume par Monte-Carlo, puis chiffrer sa précision.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Aucun après une filière scientifique ; les chapitres 2 et 3 suffisent.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 4.1 | Aire d'une plaque percée | 10 min |
| 4.2 | Volume d'une bague | 10 min |
| 4.3 | Combien de tirages pour un dixième de pour cent ? (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [ ]:
import numpy as np


def jeu_monte_carlo(cotes, tirages, rng):
    """Jeu fonctionnel d'une chaîne de cotes, par tirage : somme signée de cotes de loi normale.

    `cotes` est une liste de (nominale, tolérance, signe). Chaque cote suit une loi normale d'écart-type
    tolérance / 3 (la tolérance couvre trois écarts-types) ; le signe vaut +1 si la cote agrandit le jeu,
    -1 si elle le réduit.

    >>> rng = np.random.default_rng(0)
    >>> jeu = jeu_monte_carlo([(100.0, 0.3, 1), (60.0, 0.15, -1), (38.0, 0.15, -1)], 20000, rng)
    >>> jeu.shape
    (20000,)
    >>> round(float(jeu.mean()), 2)
    2.0
    """
    jeu = np.zeros(tirages)
    for nominale, tolerance, signe in cotes:
        jeu = jeu + signe * rng.normal(nominale, tolerance / 3, tirages)
    return jeu

### Préparation

In [ ]:
import math
import numpy as np

### Section 4.1 : un tableau NumPy tire et compte sans boucle.

In [ ]:
# notion: tableau
mesures = np.array([12.1, 11.9, 12.4, 12.0])
print("tableau :", mesures, " forme", mesures.shape, " taille", mesures.size, " type", mesures.dtype)
pas_regulier = np.linspace(0.0, 1.0, 5)
par_pas = np.arange(0.0, 10.0, 2.5)
rempli = np.full(3, 7.0)
assert mesures.shape == (4,) and mesures.size == 4 and mesures.dtype == np.float64
assert (
    np.array_equal(pas_regulier, [0.0, 0.25, 0.5, 0.75, 1.0]) and np.array_equal(par_pas, [0.0, 2.5, 5.0, 7.5])
)
assert np.array_equal(rempli, [7.0, 7.0, 7.0])

In [ ]:
# notion: sansboucle
rayons = np.array([5.0, 10.0, 15.0])
aires = np.pi * rayons**2                              # une opération sur les trois éléments à la fois
assert np.allclose(aires, [78.5398, 314.1593, 706.8583], atol=1e-4)
assert (
    np.array_equal(np.sqrt(np.array([4.0, 9.0])), [2.0, 3.0]) and np.array_equal(np.abs(np.array([-2.0])), [2.0])
)
assert (
    abs(np.log(math.e) - 1.0) < 1e-12 and abs(np.cos(0.0) - 1.0) < 1e-12 and abs(np.sin(0.0)) < 1e-12
)
assert np.array_equal(np.round(aires, 1), [78.5, 314.2, 706.9])

In [ ]:
# notion: reducs-numpy
assert mesures.sum() == np.sum(mesures) and abs(mesures.mean() - 12.1) < 1e-12
assert mesures.max() == 12.4 and mesures.min() == 11.9 and np.mean(mesures) == mesures.mean()
assert int(np.argmin(mesures)) == 1 and np.array_equal(np.cumsum(np.array([1, 2, 3])), [1, 3, 6])
# notion: alea
rng_demo = np.random.default_rng(7)
u = rng_demo.uniform(0.0, 1.0, 1000)
g = rng_demo.normal(10.0, 2.0, 1000)
assert 0.0 <= u.min() and u.max() <= 1.0 and abs(g.mean() - 10.0) < 0.3
rng_a, rng_b = np.random.default_rng(7), np.random.default_rng(7)
assert (rng_a.uniform(0, 1, 5) == rng_b.uniform(0, 1, 5)).all(), "même graine, mêmes tirages"
print("graine 7 : deux générateurs de même graine donnent les mêmes tirages")

## Exercice 4.1 : Aire d'une plaque percée (10 min)

Une plaque de $100 \times 60$ mm est percée de deux trous de $20$ mm de diamètre, centrés en $(30\,;\,30)$ et $(70\,;\,30)$. Estimez son aire en tirant des points au hasard dans le rectangle et en comptant ceux qui tombent dans la matière. Fixez la graine à $2026$ avec `default_rng`, gardez le même générateur pour des tailles successives de $1\,000$, $10\,000$, $100\,000$ puis $1\,000\,000$ tirages, et comparez le dernier résultat à l'aire exacte.

**Résultat attendu.** Aire exacte de 5 371,7 mm² ; avec un million de tirages, 5 371,5 mm², soit un écart de $-0{,}1$ mm² (calculé avant arrondi), pour un écart-type attendu de 1,8 mm² (formule à la section 4.3).

**Seconde voie.** L'aire exacte est celle du rectangle moins deux disques de rayon $10$ mm ; une autre graine donne une valeur voisine.

**Indication.** L'aire exacte est celle du rectangle moins deux disques de rayon $10$ mm. Pour chaque taille, tirez les abscisses et les ordonnées d'un coup avec `uniform`. Un masque de booléens repère les points dans un trou ; son complément repère la matière. La moyenne du complément est la proportion de points utiles : multipliée par l'aire du rectangle, elle donne l'estimation.

Les contrôles de la cellule lisent les variables `aire_exacte`, `aire_million`, `sigma_million` : gardez ces noms.

In [ ]:
L, H, R = 100.0, 60.0, 10.0                     # longueur, largeur, rayon des trous, en mm
# Votre code :
assert abs(aire_exacte - 5371.7) < 0.05, "aire exacte attendue : 5371,7 mm²"
assert abs(aire_million - aire_exacte) < 3 * sigma_million, "estimation hors de 3 écarts-types"
print("Exercice 4.1 : contrôles passés")

### Section 4.2 : un masque combiné mesure un volume.

In [ ]:
# notion: masque
valeurs = np.array([3.0, 8.0, 12.0, np.nan, 7.0])
grand = valeurs > 5.0
petit = valeurs < 10.0
assert (grand & petit).sum() == 2 and (grand | petit).sum() == 4 and (~grand).sum() == 2
assert np.where(valeurs > 5.0, 1, 0).tolist() == [0, 1, 1, 0, 1]
assert np.isin(valeurs, [3.0, 7.0]).tolist() == [True, False, False, False, True]
assert np.isnan(valeurs).any() and not np.isnan(valeurs).all()

In [ ]:
# notion: vuecopie
base = np.array([1.0, 2.0, 3.0, 4.0])
vue = base[1:3]                                        # une vue : la même mémoire
copie = base[1:3].copy()                               # une copie : une mémoire nouvelle
vue[0] = 99.0
assert base[1] == 99.0 and copie[0] == 2.0, "la vue modifie l'original, la copie non"
print("vue modifiée :", base.tolist(), " copie intacte :", copie.tolist())

In [ ]:
# notion: forme
grille = np.arange(6)
assert grille.reshape(2, 3).shape == (2, 3) and grille.reshape(2, 3).T.shape == (3, 2)
assert (
    grille.reshape(2, 3).ravel().tolist() == grille.tolist() and grille.astype(float).dtype == np.float64
)
assert grille.reshape(-1, 2).shape == (3, 2) and grille.tolist() == [0, 1, 2, 3, 4, 5]

## Exercice 4.2 : Volume d'une bague (10 min)

Une bague a un diamètre extérieur de $40$ mm, un diamètre intérieur de $20$ mm et une épaisseur de $10$ mm. Estimez son volume avec un million de tirages (graine $2026$) dans le carré de $40$ mm de côté, centré sur l'axe, puis multipliez par l'épaisseur. Comparez à la valeur exacte.

**Résultat attendu.** Volume exact de 9 425 mm³ ; estimation de 9 438 mm³, pour un écart-type attendu de 7,9 mm³ (formule à la section 4.3).

**Seconde voie.** La bague occupe la fraction $\tfrac{\pi}{4}\bigl(1 - (d/D)^2\bigr)$ du carré, avec $d$ et $D$ les diamètres : la moyenne du masque s'en approche.

**Indication.** Un point est dans la bague si son rayon au carré est compris entre le carré du rayon intérieur et celui du rayon extérieur. Les deux conditions se combinent avec `&` dans un seul masque. Le volume estimé est le volume du prisme englobant multiplié par la moyenne du masque.

Les contrôles de la cellule lisent les variables `sigma_v`, `volume_exact`, `volume_mc` : gardez ces noms.

In [ ]:
D_ext, D_int, e = 40.0, 20.0, 10.0
# Votre code :
assert round(volume_exact) == 9425, "volume exact attendu : 9425 mm³"
assert abs(volume_mc - volume_exact) < 3 * sigma_v, "estimation hors de 3 écarts-types"
print("Exercice 4.2 : contrôles passés")

### Section 4.3 : la précision croît comme la racine de N.

In [ ]:
# notion: std-ddof
echantillon = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
assert echantillon.std() == 2.0, "écart-type de la population"
assert abs(echantillon.std(ddof=1) - 2.13809) < 1e-5, "écart-type de l'échantillon (n - 1)"
print(f"écart-type de la population {echantillon.std():.4f}, de l'échantillon {echantillon.std(ddof=1):.4f}")

## Exercice 4.3 : Combien de tirages pour un dixième de pour cent ? (défi, 25 min)

Combien de tirages faut-il pour connaître l'aire de la plaque à $0{,}1$ % près, au sens de l'écart-type relatif ? Répétez $200$ fois l'estimation avec $10\,000$ tirages (graine $2026$), mesurez la dispersion relative, puis extrapolez : l'écart-type décroît comme l'inverse de la racine du nombre de tirages. Ensuite, dans la chaîne de cotes du chapitre, quelle proportion des jeux est inférieure à $1{,}6$ mm ? Comparez-la à celle de la loi normale, obtenue avec `math.erfc`.

**Résultat attendu.** Écart-type relatif mesuré de 0,359 % à $10\,000$ tirages ; environ 129 000 tirages en extrapolant, à 13 000 près (10 % d'incertitude, due aux 200 répétitions) ; pour la chaîne de cotes, 0,057 % des jeux sous $1{,}6$ mm, contre 0,055 % par la loi normale.

**Seconde voie.** La formule de précision donne 117 000 tirages, soit un écart de 1,0 fois l'incertitude de l'extrapolation ; la loi normale du jeu a pour écart-type 0,1225 mm, d'où $z = (1{,}6 - 2)/\sigma$ et la proportion cherchée par $\tfrac12\,\mathrm{erfc}(-z/\sqrt{2})$.

**Indication.** Stockez les $200$ estimations dans un tableau, puis divisez leur écart-type (avec `ddof=1`) par l'aire exacte. Pour passer de cet écart-type relatif à la cible de $0{,}1$ %, multipliez le nombre de tirages par le carré du rapport des deux écarts-types. Un écart-type estimé sur $n$ répétitions est lui-même incertain, d'environ $1/\sqrt{2(n-1)}$ en valeur relative ; le nombre de tirages, proportionnel à son carré, l'est deux fois plus. Pour la chaîne de cotes, la proportion est la moyenne du masque `jeu < 1.6`, et la loi normale donne $\tfrac12\,\mathrm{erfc}(-z/\sqrt{2})$ avec $z = (1{,}6 - 2)/\sigma$, $\sigma$ étant le tiers de la somme quadratique.

In [ ]:
# Votre code :
assert abs(N_cible - N_theorie) < 3 * incertitude * N_theorie, "hors de 3 incertitudes"
# Votre code :
assert (
    abs(tol_pire_cas - 0.60) < 1e-12 and abs(tol_quadratique - 0.367) < 0.0005
), "tolérances attendues"
assert abs(jeu_moyen - 2.0) < 5 * sigma_jeu / math.sqrt(N_chaine), "jeu moyen attendu : 2,0 mm"
assert abs(sigma_mesure - sigma_jeu) < 0.002, "écart-type du jeu attendu : 0,1225 mm"
assert ecart_proportion < 4.0, "proportion hors de 4 écarts-types de la théorie"
# Votre code :
assert np.array_equal(jeu_module, jeu), "le module et le script tirent les mêmes valeurs"
print("Exercice 4.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F04-1 : Tirages aléatoires (`numpy.random.default_rng`)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)           # graine fixée
alesage = rng.normal(50.05, 0.01, 10**5)  # diamètres, mm
arbre = rng.normal(50.00, 0.02, 10**5)    # diamètres, mm
jeu = alesage - arbre                     # jeu diamétral, mm
p_serrage = np.mean(jeu < 0.0) * 100      # proportion de serrage
print(f"Serrage : {p_serrage:.2f} %".replace(".", ","))
assert 0.0 < p_serrage < 5.0              # contrôle physique